# Part 2 — Correlation-Based / Factor-Based Multi-Omic Integration with MOFA

In this notebook we use **Multi-Omics Factor Analysis (MOFA)** to integrate transcriptomics, proteomics, and methylation data from TCGA-BRCA.

MOFA is an unsupervised latent factor model: it learns a compact representation of patients from multiple molecular views without using the subtype labels. We then ask whether those learned factors separate or predict breast cancer subtypes.

## Learning objectives

By the end of this notebook you should be able to:

- Load the already processed and aligned omics tables.
- Use one shared train/test split for every downstream comparison.
- Fit MOFA on the multi-omic data to learn shared latent factors.
- Use the learned factors for simple subtype prediction.
- Inspect which factors are associated with subtype and which omics views contribute to those factors.

The output of this notebook is a set of MOFA factors, subtype predictions, and metrics that can be carried into the next workshop section.

## Expected data layout

```text
../../data_tmp/TCGA-BRCA/
├── transcriptomics.pkl
├── proteomics.pkl
└── methylation.pkl
```

Each table is assumed to be already processed, aligned, and complete.

Assumptions used in this notebook:

- rows are patients
- the patient ID is stored in the index
- `subtype` is the prediction target
- all remaining columns are molecular features
- there is no missing patient data
- there is no imputation or preprocessing step in this tutorial

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import train_test_split

# MOFA is implemented in mofapy2.
# Install once in your workshop environment if needed:
# !pip install mofapy2
try:
    from mofapy2.run.entry_point import entry_point
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "mofapy2 is not installed. Run `pip install mofapy2` in this environment, "
        "restart the kernel, and rerun the notebook."
    ) from exc

RANDOM_STATE = 42
DATA_DIR = Path("../../data_tmp/TCGA-BRCA")
TARGET_COLUMN = "subtype"

OMICS_FILES = {
    "transcriptomics": DATA_DIR / "transcriptomics.pkl",
    "proteomics": DATA_DIR / "proteomics.pkl",
    "methylation": DATA_DIR / "methylation.pkl",
}

TEST_SIZE = 0.25
N_FACTORS = 10
MOFA_ITERATIONS = 500

## 1. Load the already processed omics tables

No alignment, filtering, imputation, or feature processing is performed here. The checks below only verify that the notebook assumptions hold before fitting MOFA.

In [ ]:
omics = {name: pd.read_pickle(path) for name, path in OMICS_FILES.items()}

for name, df in omics.items():
    print(f"{name:15s}: {df.shape[0]:4d} patients × {df.shape[1] - 1:6d} molecular features")
    assert TARGET_COLUMN in df.columns, f"{name} is missing the '{TARGET_COLUMN}' column"

# The target is stored in each table; use the first table as the reference.
reference_name = next(iter(omics))
patient_ids = omics[reference_name].index.astype(str)
y = omics[reference_name][TARGET_COLUMN].astype(str)

# Confirm the workshop assumptions without changing the data.
for name, df in omics.items():
    assert df.index.astype(str).equals(patient_ids), f"Patient index differs in {name}"
    assert df[TARGET_COLUMN].astype(str).equals(y), f"Subtype labels differ in {name}"

X_omics = {
    name: df.drop(columns=[TARGET_COLUMN])
    for name, df in omics.items()
}

print("
Subtype counts:")
display(y.value_counts().rename("n_patients").to_frame())

## 2. Create one shared train/test split

The split is created once using the patient IDs and reused everywhere. MOFA is unsupervised, but the same split is still used for downstream subtype prediction from the learned factors.

In [ ]:
train_ids, test_ids = train_test_split(
    patient_ids,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

train_ids = pd.Index(train_ids, name="patient_id")
test_ids = pd.Index(test_ids, name="patient_id")

X_train_omics = {name: X.loc[train_ids] for name, X in X_omics.items()}
X_test_omics = {name: X.loc[test_ids] for name, X in X_omics.items()}

y_train = y.loc[train_ids]
y_test = y.loc[test_ids]

print(f"Training patients: {len(train_ids)}")
print(f"Test patients:     {len(test_ids)}")

## 3. Helper functions for evaluation

The classifier is intentionally simple: multinomial logistic regression on top of MOFA factors. This keeps the focus on the representation learned by MOFA, not on classifier complexity.

In [ ]:
def evaluate_predictions(y_true, y_pred, model_name):
    """Return common classification metrics in one row."""
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
    }


def fit_factor_classifier(factors_df, y, train_ids, test_ids, model_name):
    """Fit a classifier on training factors and evaluate on held-out test factors."""
    clf = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    clf.fit(factors_df.loc[train_ids], y.loc[train_ids])
    pred = clf.predict(factors_df.loc[test_ids])
    metrics = evaluate_predictions(y.loc[test_ids], pred, model_name)
    return clf, pred, metrics

## 4. Prepare the MOFA input

`mofapy2` accepts a list of views, where each view is a list of groups. We have one biological group and three views:

- transcriptomics
- proteomics
- methylation

MOFA receives only molecular features. The `subtype` labels are not used to train the factor model.

In [ ]:
def build_mofa_matrix_input(X_by_omic):
    """Convert aligned omics DataFrames to the nested matrix format expected by mofapy2."""
    data = []
    view_names = []
    feature_names = []

    for view_name, X in X_by_omic.items():
        view_names.append(view_name)
        feature_names.append(X.columns.astype(str).tolist())
        data.append([X.to_numpy(dtype=np.float32)])  # one group, many views

    sample_names = [patient_ids.astype(str).tolist()]  # one group
    group_names = ["TCGA-BRCA"]

    return data, view_names, feature_names, sample_names, group_names

mofa_data, view_names, feature_names, sample_names, group_names = build_mofa_matrix_input(X_omics)

for view_name, matrix in zip(view_names, mofa_data):
    print(f"{view_name:15s}: {matrix[0].shape}")

## 5. Fit MOFA

MOFA learns latent factors that explain covariance structure across the molecular views. Because this is unsupervised, the target labels are not passed to the model.

For a workshop, `MOFA_ITERATIONS` is set to a moderate value so that the notebook remains practical to run. Increasing it may improve convergence in a production analysis.

In [ ]:
def fit_mofa(data, view_names, feature_names, sample_names, group_names):
    """Fit a compact MOFA model and return the trained entry point object."""
    model = entry_point()

    # These are MOFA's internal centering/scaling controls, not external preprocessing.
    # scale_views=True helps prevent one view from dominating only because it has a larger scale.
    model.set_data_options(
        scale_views=True,
        scale_groups=False,
        center_groups=True,
        use_float32=True,
    )

    model.set_data_matrix(
        data=data,
        likelihoods=["gaussian"] * len(view_names),
        views_names=view_names,
        groups_names=group_names,
        samples_names=sample_names,
        features_names=feature_names,
    )

    model.set_model_options(
        factors=N_FACTORS,
        spikeslab_factors=False,
        spikeslab_weights=True,
        ard_factors=False,
        ard_weights=True,
    )

    model.set_train_options(
        iter=MOFA_ITERATIONS,
        convergence_mode="fast",
        seed=RANDOM_STATE,
        verbose=False,
        quiet=True,
    )

    model.build()
    model.run()
    return model

mofa_model = fit_mofa(mofa_data, view_names, feature_names, sample_names, group_names)

## 6. Extract MOFA factors

The factor matrix has one row per patient and one column per latent factor. These factors are the integrated representation that we will use for subtype prediction and visualization.

In [ ]:
def extract_mofa_factors(model, patient_ids):
    """Extract factor expectations from a fitted mofapy2 model."""
    Z = model.model.nodes["Z"].getExpectations()["E"]
    factor_names = [f"MOFA_factor_{i + 1}" for i in range(Z.shape[1])]
    return pd.DataFrame(Z, index=patient_ids.astype(str), columns=factor_names)

factors_df = extract_mofa_factors(mofa_model, patient_ids)
factors_df["subtype"] = y.values

print(factors_df.shape)
display(factors_df.head())

## 7. Predict subtype from MOFA factors

This evaluates whether the unsupervised multi-omic factors contain subtype-relevant information. The classifier sees labels only on the training patients and is evaluated on the held-out test patients.

In [ ]:
feature_cols = [c for c in factors_df.columns if c.startswith("MOFA_factor_")]

mofa_factor_clf, mofa_pred, mofa_metrics = fit_factor_classifier(
    factors_df[feature_cols],
    y,
    train_ids.astype(str),
    test_ids.astype(str),
    "MOFA factors + logistic regression",
)

results_df = pd.DataFrame([mofa_metrics]).sort_values("balanced_accuracy", ascending=False)
display(results_df)

print(classification_report(y_test, mofa_pred))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    mofa_pred,
    xticks_rotation=45,
)
plt.title("Confusion matrix — MOFA factors")
plt.tight_layout()
plt.show()

## 8. Visualize the first two MOFA factors

A two-factor plot is not a full model evaluation, but it is a useful workshop diagnostic. Strong subtype structure here means the unsupervised latent space has captured subtype-associated biology.

In [ ]:
plot_df = factors_df.loc[test_ids.astype(str), ["MOFA_factor_1", "MOFA_factor_2", "subtype"]]

fig, ax = plt.subplots(figsize=(7, 5))

for subtype, group in plot_df.groupby("subtype"):
    ax.scatter(
        group["MOFA_factor_1"],
        group["MOFA_factor_2"],
        label=subtype,
        alpha=0.8,
    )

ax.axhline(0, linewidth=0.8)
ax.axvline(0, linewidth=0.8)
ax.set_title("Held-out patients in MOFA latent space")
ax.set_xlabel("MOFA factor 1")
ax.set_ylabel("MOFA factor 2")
ax.legend(title="Subtype", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 9. Which MOFA factors are associated with subtype?

MOFA is unsupervised, so factor labels are arbitrary. This small diagnostic ranks factors by how much their values differ across subtypes in the training set.

In [ ]:
def eta_squared_by_factor(factor_table, labels):
    """Compute one-way ANOVA eta-squared for each factor without fitting a predictive model."""
    rows = []
    labels = labels.astype(str)

    for factor in factor_table.columns:
        values = factor_table[factor]
        grand_mean = values.mean()
        ss_between = 0.0
        ss_total = ((values - grand_mean) ** 2).sum()

        for subtype, idx in labels.groupby(labels).groups.items():
            group_values = values.loc[idx]
            ss_between += len(group_values) * (group_values.mean() - grand_mean) ** 2

        eta2 = ss_between / ss_total if ss_total > 0 else np.nan
        rows.append({"factor": factor, "eta_squared": eta2})

    return pd.DataFrame(rows).sort_values("eta_squared", ascending=False)

factor_subtype_assoc = eta_squared_by_factor(
    factors_df.loc[train_ids.astype(str), feature_cols],
    y_train,
)

display(factor_subtype_assoc)

## 10. Inspect view contributions through MOFA weights

MOFA learns weights for each view and factor. A high absolute weight means that a feature contributes strongly to that factor. Here we summarize the average absolute weight per omic view and factor.

In [ ]:
def extract_view_weight_summary(model, view_names):
    """Summarize average absolute MOFA weights by view and factor."""
    W = model.model.nodes["W"].getExpectations()
    rows = []

    for view_name, view_weights in zip(view_names, W):
        weight_matrix = view_weights["E"]  # features × factors
        for k in range(weight_matrix.shape[1]):
            rows.append(
                {
                    "view": view_name,
                    "factor": f"MOFA_factor_{k + 1}",
                    "mean_abs_weight": np.mean(np.abs(weight_matrix[:, k])),
                }
            )

    return pd.DataFrame(rows)

view_weight_summary = extract_view_weight_summary(mofa_model, view_names)
view_weight_pivot = view_weight_summary.pivot(
    index="factor",
    columns="view",
    values="mean_abs_weight",
)

display(view_weight_pivot)

In [ ]:
# Plot the factors most associated with subtype and show their average absolute weights by view.
top_factors = factor_subtype_assoc.head(5)["factor"].tolist()

ax = view_weight_pivot.loc[top_factors].plot(kind="bar", figsize=(9, 4))
ax.set_title("Average absolute MOFA weight by view for subtype-associated factors")
ax.set_xlabel("MOFA factor")
ax.set_ylabel("Mean absolute weight")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Save outputs for the next workshop section

These files provide a clean handoff:

- performance metrics
- patient-level predictions on the held-out split
- MOFA factors for all patients
- factor/subtype association diagnostics
- view-level weight summaries

In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

predictions_df = pd.DataFrame(
    {
        "patient_id": test_ids.astype(str),
        "true_subtype": y_test.values,
        "mofa_factor_prediction": mofa_pred,
    }
).set_index("patient_id")

factors_to_save = factors_df.copy()
factors_to_save["split"] = "train"
factors_to_save.loc[test_ids.astype(str), "split"] = "test"

results_df.to_csv(OUTPUT_DIR / "part2_mofa_metrics.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "part2_mofa_predictions.csv")
factors_to_save.to_csv(OUTPUT_DIR / "part2_mofa_factors.csv")
factor_subtype_assoc.to_csv(OUTPUT_DIR / "part2_mofa_factor_subtype_associations.csv", index=False)
view_weight_summary.to_csv(OUTPUT_DIR / "part2_mofa_view_weight_summary.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())

## Takeaways

- MOFA learns a shared multi-omic latent space without using subtype labels.
- The train/test split is reused for subtype prediction from the learned factors.
- If MOFA factors predict subtype well, the integrated molecular covariance structure is subtype-informative.
- Factor visualizations help participants see whether subtype structure emerges naturally.
- View-level weights help explain which omics contribute to the most subtype-associated factors.
- The saved MOFA factors and predictions are ready for comparison with the next integration strategy in the workshop.